In [17]:
from typing import List

import nltk
from nltk import pos_tag_sents, sent_tokenize, word_tokenize
from nltk.stem.wordnet import WordNetLemmatizer
from pygments.lexer import words
from nltk.corpus import wordnet

nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/qr/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [45]:
from typing import Tuple


def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # Default to noun


text = "Python programmers often tend like programming sdgfgsdfg in python because it's like english. We call people who program in python pythonistas"

sentences = sent_tokenize(text)
tokens = [word_tokenize(sent) for sent in sentences]
positions: List[List[Tuple[str, str]]] = pos_tag_sents(words)

lemma = WordNetLemmatizer()
lemma_words = [
    lemma.lemmatize(word, get_wordnet_pos(pos)).lower()
    for sent in positions
    for word, pos in sent
    if wordnet.synsets(word)
]

print(text)
# print(sentences)
# print(words)
# print(positions)
print(" ".join(lemma_words))

Python programmers often tend like programming sdgfgsdfg in python because it's like english. We call people who program in python pythonistas
python programmer often tend like program in python it like english call people who program in python


In [8]:
import polars as pl

ai_type = pl.Enum(["AI", "UNKNOWN", "HUMAN"])
blacklist = [
    "Please wait while your request is being verified...",
    "AI scrapers break the web, to use this page you'll need JavaScript enabled.",
    "Javascript is required for this site. Consider enabling Javascript or upgrading to a modern browser.",
    "This domain name registration has expired and renewal or deletion are pending. If you are the registrant and "
    "want to renew the domain name, please contact your registration service provider.",
    "We're sorry but doesn't work properly without JavaScript enabled. Please enable it to continue.",
    "This page was generated on 20"
]

(
    pl
    .read_parquet("/mnt/Fast2T/data/stage_4/CC-MAIN-20260206181458-20260206211458-00190.parquet")
    .filter(pl.col("lang_en") > 0.7)
    .filter(pl.col("text").str.len_bytes() > 200)
    .filter(~pl.col("text").str.contains_any(blacklist))
    .with_columns(
        pl
        .when(pl.col('ai') < 0.005).then(pl.lit("HUMAN"))
        .when(pl.col('ai') > 0.9).then(pl.lit("AI"))
        .otherwise(pl.lit("UNKNOWN"))
        .cast(ai_type).alias("is_ai"),
    )
    .with_columns(
        pl
        .col("outflow")
        .map_elements(
            lambda s: [{ "url": k, "occurrences": v } for k, v in s.items() if v is not None],
            return_dtype=pl.List(pl.Struct({ "url": pl.String, "occurrences": pl.Int64 })),
        ),
    )
    .select("host", "url", "text", "outflow", "is_ai")
)

host,url,text,outflow,is_ai
str,str,str,list[struct[2]],enum
"""03-128-53-208.plesk.page""","""http://03-128-53-208.plesk.pag…","""Day Two Draft Recap: Cubs Add …","[{""03-128-53-208.plesk.page"",41}, {""www.facebook.com"",7}, … {""flipboard.com"",1}]","""HUMAN"""
"""550squadronassociation.org.uk""","""http://550squadronassociation.…","""Listed below is the informatio…","[{""550squadronassociation.org.uk"",13}]","""UNKNOWN"""
"""allmedschools.com""","""http://allmedschools.com/unive…","""AMCAS application accepted? Ye…","[{""wordpress.org"",1}, {""allmedschools.com"",5}]","""HUMAN"""
"""anarchy2023.org""","""http://anarchy2023.org/en/book…","""Bookfair/Exposition You can ge…","[{""anarchy2023.org"",39}]","""HUMAN"""
"""arahn.100webspace.net""","""http://arahn.100webspace.net/s…","""ARAHN A_R_A_H_N FAQ Search Mem…","[{""arahn.100webspace.net"",10}, {""www.phpbb.com"",1}, … {""www.idatingonline.com"",1}]","""HUMAN"""
…,…,…,…,…
"""zarzarfashion.com""","""https://zarzarfashion.com/prod…","""What It Is: An anti-aging faci…","[{""www.facebook.com"",1}, {""www.instagram.com"",1}, … {""click.linksynergy.com"",49}]","""HUMAN"""
"""zecatech.com""","""https://zecatech.com/project/e…","""On the other hand, we denounce…",[],"""UNKNOWN"""
"""zenerva.com""","""https://zenerva.com/app-reskin…","""Designers lose time, cash, and…","[{""www.facebook.com"",1}, {""twitter.com"",1}, … {""pinterest.com"",1}]","""HUMAN"""


In [5]:
import polars as pl

(
    pl
    .scan_parquet("/mnt/Fast2T/data/stage_4/*.parquet")
    .select("text")
    .with_columns(abfff=(pl.col("text").str.len_bytes() / 100.0).cast(pl.Int64))
    .group_by("abfff")
    .len()
    .collect()
)

abfff,len
i64,u32
3671,1
38678,2
8914,3
4594,2
1447,13
…,…
7988,1
914,114
780,77
